<a href="https://colab.research.google.com/github/SheffieldML/GPc/blob/main/python/examples/olympic_marathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Olympic marathon with `gpc` + `pods`

CIP-001D Phase E + CIP-0031. Modelling uses core `gpc` only; the dataset comes from external [`pods`](https://pypi.org/project/pods/).

`plot_1d` defaults to `predict` (\(y_*\)). Use `noiseless=True` for `predict_noiseless` (\(f_*\)).

```bash
pip install pods matplotlib   # or: pip install 'gpc[examples]'
```

In [ ]:
%pip install --no-cache-dir --upgrade --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "gpc==0.1.0a4"


In [ ]:
%pip install pods matplotlib ipywidgets


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import sys
from pathlib import Path

root = Path.cwd()
if (root / "gpc").is_dir():
    sys.path.insert(0, str(root))
elif (root.parent / "gpc").is_dir():
    sys.path.insert(0, str(root.parent))

import numpy as np
import gpc

try:
    import pods
except ImportError as exc:
    raise SystemExit("pip install pods  (or gpc[examples])") from exc

data = pods.datasets.olympic_marathon_men()
X = np.asarray(data["X"], dtype=float)
Y = np.asarray(data["Y"], dtype=float)
if X.ndim == 1:
    X = X[:, None]
if Y.ndim == 1:
    Y = Y[:, None]

m = gpc.GPRegression(X, Y)
m.set_by_path("rbf.lengthscale", 30.0)
m.optimize(iters=100, messages=True)
print("objective:", m.objective())
print({p: m.get_by_path(p) for p in m.param_paths()})

In [ ]:
from gpc.plot import plot_1d
import matplotlib.pyplot as plt

mu_y, var_y = m.predict(X)
mu_f, var_f = m.predict_noiseless(X)
sigma2 = m.get_by_path("likelihood.variance")
print("likelihood.variance:", sigma2)
print("mean(var_y - var_f) at train X:", float(np.mean(var_y - var_f)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
plot_1d(m, X, Y, ax=axes[0], label="y_*", noiseless=False)
axes[0].set_title("predict (y_*)")
axes[0].set_xlabel("year")
axes[0].set_ylabel("time (hours)")
plot_1d(m, X, Y, ax=axes[1], label="f_*", noiseless=True)
axes[1].set_title("predict_noiseless (f_*)")
axes[1].set_xlabel("year")
plt.tight_layout()
plt.show()

Sparse DTCVAR variant (optional): place inducing inputs on a year grid.

**Init.** Years are \(O(2000)\) with gaps of decades. Default `rbf.lengthscale=1` makes neighbouring inducing points look almost identical, so **DTC** can hit a non-positive-definite `K_uu` and **DTCVAR** can "succeed" into a flat, high-noise fit. Set the lengthscale in years (same as the FTC cell). `rbf.variance=1` is fine once the lengthscale matches the data scale — bumping variance alone does not fix the conditioning.

In [ ]:
Z = np.linspace(float(X.min()), float(X.max()), 8)[:, None]
ms = gpc.SparseGPRegression(X, Y, Z)  # approx='dtcvar' default
ms.set_by_path("rbf.lengthscale", 30.0)  # year-scale; variance=1 is OK
ms.optimize(iters=80, messages=True)
print("sparse objective:", ms.objective(), "approx:", ms.approx)
print({p: ms.get_by_path(p) for p in ms.param_paths() if "inducing" not in p})
mu_y, var_y = ms.predict(X)
mu_f, var_f = ms.predict_noiseless(X)
print("sparse mean(var_y - var_f):", float(np.mean(var_y - var_f)),
      "likelihood.variance:", ms.get_by_path("likelihood.variance"))
plot_1d(ms, X, Y, label="dtcvar y_*")
plt.show()

# Optional: DTC with the same init (default ls=1 often raises NPD on this grid).
# md = gpc.SparseGPRegression(X, Y, Z, approx="dtc")
# md.set_by_path("rbf.lengthscale", 30.0)
# md.optimize(iters=80, messages=True)